# 🚀 Elite Customer Churn Prediction Project

## 🎓 Executive Summary
This project develops a predictive model for customer churn in the telecommunications industry. We utilize advanced techniques to handle class imbalance, engineer features, and provide cutting-edge **Explainable AI (XAI)** insights using SHAP.

## 1. Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.drop('customerID', axis=1, inplace=True)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.head()

## 3. Handling Class Imbalance (SMOTE)
Because Churn is naturally imbalanced, we use **Synthetic Minority Over-sampling Technique (SMOTE)** to generate artificial churn examples so the model learns equally.

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns

encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: \n{y_train.value_counts()}")
print(f"After SMOTE: \n{pd.Series(y_train_smote).value_counts()}")

## 4. Hyperparameter Tuning (GridSearchCV)
To ensure optimal performance, we tune the Random Forest's parameters.

In [ ]:
rf = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10]
}
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_smote, y_train_smote)

best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

## 5. Model Evaluation

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 6. Explainable AI (SHAP)
Instead of a black-box model, we calculate SHAP values to explain EXACTLY which features drive customer churn at this telecommunications provider.

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[1], X_test, plot_type="bar")